In [78]:
import sys
sys.path.append('.')
import sqlite3 as sql3
from utils.config import DB_PATH
from utils.db import get_price
from win_rates import get_win_rates
from datetime import datetime
import pytz
import re

In [79]:
win_rates = get_win_rates()
good_wallets = {w['wallet'] for w in win_rates if w['total'] > 100 and w['win_rate'] > 50}

con = sql3.connect(DB_PATH)
con.row_factory = sql3.Row
cur = con.cursor()

wallets = list(good_wallets)
placeholders = ','.join(['?' for _ in wallets])
cur.execute(f"""
    SELECT * FROM trades 
    WHERE wallet IN ({placeholders}) 
    AND title LIKE 'Bitcoin%'
    AND title LIKE '%Up or Down%'
    AND side = 'BUY'
    AND timestamp >= 1761955200
    ORDER BY wallet, condition_id, timestamp ASC
""", wallets)

rows = cur.fetchall()
trades = [dict(row) for row in rows]
con.close()

In [80]:
con = sql3.connect(DB_PATH)
cur = con.cursor()

cur.execute("SELECT MIN(timestamp), MAX(timestamp) FROM prices WHERE symbol='BTCUSDT'")\

a = cur.fetchall()

con.close()

In [47]:
a

[(1770915900000, 1772392380000)]

In [81]:
ET = pytz.timezone('US/Eastern')

def parse_end_time(title, buy_timestamp):
    # "Bitcoin Up or Down - February 16, 2:15PM-2:30PM ET" -> end is 2:30PM
    # "Bitcoin Up or Down - February 16, 1PM ET" -> end is 1PM
    # "Bitcoin Up or Down on February 22?" -> skip, daily market
    
    if 'on February' in title or 'on March' in title:
        return None  # daily markets, skip
    
    try:
        # Extract date
        date_match = re.search(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d+),\s+', title)
        if not date_match:
            return None
        month_str = date_match.group(1)
        day = int(date_match.group(2))
        year = 2026
        month = datetime.strptime(month_str, '%B').month

        # Extract end time — look for range first, then single time
        range_match = re.search(r'(\d+:\d+(?:AM|PM))-(\d+:\d+(?:AM|PM))', title)
        single_match = re.search(r'(\d+(?::\d+)?(?:AM|PM))\s+ET', title)

        if range_match:
            end_str = range_match.group(2)
        elif single_match:
            end_str = single_match.group(1)
        else:
            return None

        # Parse end time
        if ':' in end_str:
            end_dt = datetime.strptime(f"{month}/{day}/{year} {end_str}", '%m/%d/%Y %I:%M%p')
        else:
            end_dt = datetime.strptime(f"{month}/{day}/{year} {end_str}", '%m/%d/%Y %I%p')

        end_dt = ET.localize(end_dt)
        return int(end_dt.timestamp())

    except Exception as e:
        return None


In [82]:
seen = {}
entries = []
for trade in trades:
    key = (trade['wallet'], trade['condition_id'])
    if key not in seen:
        seen[key] = trade
        entries.append(trade)

In [83]:
from collections import defaultdict
direction_check = defaultdict(set)
for trade in trades:
    key = (trade['wallet'], trade['condition_id'])
    direction_check[key].add(trade['outcome'])

In [84]:
entries = [e for e in entries if len(direction_check[(e['wallet'], e['condition_id'])]) == 1]


In [85]:
TRADE_SIZE = 100  # fixed $100 per trade
results = []
skipped = 0

for entry in entries:
    buy_ts = int(entry['timestamp'])
    end_ts = parse_end_time(entry['title'], buy_ts)
    
    if end_ts is None:
        skipped += 1
        print(f"SKIP - parse failed: {entry['title']}")
        continue
    
    if end_ts <= buy_ts:
        skipped += 1
        print(f"SKIP - end <= buy: {entry['title']} | buy={buy_ts} end={end_ts}")
        continue

    entry_price_row = get_price('BTCUSDT', buy_ts * 1000)
    exit_price_row = get_price('BTCUSDT', end_ts * 1000)

    if not entry_price_row or not exit_price_row:
        skipped += 1
        print(f"SKIP - no price data: {entry['title']} | buy_ts={buy_ts*1000} end_ts={end_ts*1000}")
        continue

    entry_price = entry_price_row[0][4]  # close price
    exit_price = exit_price_row[0][4]    # close price

    direction = entry['outcome']  # 'Up' or 'Down'

    if direction == 'Up':
        pnl = (exit_price - entry_price) / entry_price * TRADE_SIZE
    else:
        pnl = (entry_price - exit_price) / entry_price * TRADE_SIZE

    won = pnl > 0

    results.append({
        'wallet': entry['wallet'],
        'title': entry['title'],
        'outcome': direction,
        'buy_ts': buy_ts,
        'end_ts': end_ts,
        'entry_price': entry_price,
        'exit_price': exit_price,
        'pnl': pnl,
        'won': won
    })

SKIP - parse failed: Bitcoin Up or Down on December 23?
SKIP - parse failed: Bitcoin Up or Down on December 31?
SKIP - parse failed: Bitcoin Up or Down on December 25?
SKIP - end <= buy: Bitcoin Up or Down - February 1, 11AM ET | buy=1769961614 end=1769961600
SKIP - end <= buy: Bitcoin Up or Down - February 27, 5:00PM-5:05PM ET | buy=1772229907 end=1772229900
SKIP - end <= buy: Bitcoin Up or Down - February 27, 11:30PM-11:35PM ET | buy=1772253307 end=1772253300
SKIP - end <= buy: Bitcoin Up or Down - March 1, 11:45AM-11:50AM ET | buy=1772383825 end=1772383800
SKIP - end <= buy: Bitcoin Up or Down - March 1, 6:45AM-7:00AM ET | buy=1772366411 end=1772366400
SKIP - end <= buy: Bitcoin Up or Down - February 27, 7:05PM-7:10PM ET | buy=1772237405 end=1772237400
SKIP - end <= buy: Bitcoin Up or Down - February 28, 4:35PM-4:40PM ET | buy=1772314817 end=1772314800
SKIP - end <= buy: Bitcoin Up or Down - February 28, 7:45AM-8:00AM ET | buy=1772283635 end=1772283600
SKIP - end <= buy: Bitcoin Up 

In [86]:
total = len(results)
wins = sum(1 for r in results if r['won'])
total_pnl = sum(r['pnl'] for r in results)

print(f"Total trades: {total}")
print(f"Skipped: {skipped}")
print(f"Win rate: {wins/total*100:.1f}%")
print(f"Total PnL: ${total_pnl:.2f}")
print(f"Avg PnL per trade: ${total_pnl/total:.2f}")

Total trades: 930
Skipped: 112
Win rate: 47.3%
Total PnL: $275.67
Avg PnL per trade: $0.30


In [87]:
from collections import defaultdict
wallet_results = defaultdict(list)
for r in results:
    wallet_results[r['wallet']].append(r)

print("\n--- Per Wallet ---")
for wallet, trades_list in sorted(wallet_results.items(), key=lambda x: sum(r['pnl'] for r in x[1]), reverse=True):
    w_total = len(trades_list)
    w_wins = sum(1 for r in trades_list if r['won'])
    w_pnl = sum(r['pnl'] for r in trades_list)
    print(f"{wallet[:10]} | trades: {w_total} | win rate: {w_wins/w_total*100:.1f}% | PnL: ${w_pnl:.2f}")


--- Per Wallet ---
0x986b121c | trades: 175 | win rate: 46.3% | PnL: $220.40
0xce244e7b | trades: 57 | win rate: 57.9% | PnL: $83.40
0xd84c2b6d | trades: 205 | win rate: 52.2% | PnL: $5.51
0x5924ca48 | trades: 21 | win rate: 76.2% | PnL: $3.69
0xcc500cbc | trades: 56 | win rate: 48.2% | PnL: $3.42
0x571c285a | trades: 12 | win rate: 83.3% | PnL: $2.83
0x88f46b9e | trades: 89 | win rate: 13.5% | PnL: $1.47
0x537494c5 | trades: 18 | win rate: 61.1% | PnL: $1.18
0xf6963d4c | trades: 5 | win rate: 80.0% | PnL: $0.61
0x1979ae6b | trades: 10 | win rate: 50.0% | PnL: $0.45
0x3e3bb2ba | trades: 8 | win rate: 37.5% | PnL: $-0.52
0x873e349e | trades: 46 | win rate: 15.2% | PnL: $-1.95
0x98714cf2 | trades: 227 | win rate: 54.6% | PnL: $-4.72
0xe00740bc | trades: 1 | win rate: 0.0% | PnL: $-40.11
